In [2]:
import os
import pandas as pd
import scipy.stats as stats
import numpy as np

In [91]:
from pathlib import Path
base_dir = Path.cwd()
fpath= os.path.join(base_dir, "DATA", "cars_sampled.csv")
#read data  
cars_data = pd.read_csv(fpath)


In [92]:
cars = cars_data.copy()
#total number of records
print("Total number of records:", cars.shape[0])

Total number of records: 50001


In [93]:
# working range of data(yearOfRegistration >= 1950) & (yearOfRegistration <= 2018 ) &  (price >= 100) & (price <= 150000) & (powerps >= 10) & (powerps <= 500)
cars = cars.loc[((cars['yearOfRegistration'] >= 1950) & (cars['yearOfRegistration'] <= 2018 )) & 
                 ((cars['price'] >= 100)  & (cars['price'] <= 150000) )&
                 ((cars['powerPS'] >= 10) & (cars['powerPS'] <= 500)) ]
# total number of records after removing outliers
print("Total number of records after removing outliers:", cars.shape[0])

Total number of records after removing outliers: 43155


One Sample Test of Mean  ONE SAMPLE Z-TEST
Q: Three Years back, the average price of a used car was $ 6000. Has it Changed now?

Hypotheses Testing Steps 
Hypotheses | HO : mean = 6000 HA : mean != 6000 
Sample Statistics = 6188.34 
Test Statistics "t" value=0.815 
critical value = ?

alpha = 0.05
computed uncertainity =?
Decision =?

In [131]:
#subsample of 1000 records
sample_size = 1000
sample = cars.sample(n=sample_size, random_state=0)
#postulate mean of the population
pop_mean = 6000
#calculate sample mean and sample standard deviation
sample_mean = sample['price'].mean()
print(sample_mean)
# one sample t-test calculate t-statistic, p-value
z_statistic, p_value = stats.ttest_1samp(sample['price'], pop_mean    )
print("z-statistic:", z_statistic)          
print("p-value:", p_value)
#number of records in the sample    
n = len(cars['price'])
#number of degrees of freedom
df = n - 1
print(n,df)
# significance_level alpha
alpha = 0.05
#critical value for two-tailed test
critical_value = stats.t.ppf([alpha/2,1 - alpha/2], df)
print("Critical values:", critical_value)
print(stats.norm.cdf(0.05))
print("Z-Critical Value", stats.norm.ppf(1- alpha/2))


6188.337
z-statistic: 0.8148683326967584
p-value: 0.41534189398889065
43155 43154
Critical values: [-1.96001896  1.96001896]
0.5199388058383725
Z-Critical Value 1.959963984540054


TEST RESULT
Test Statistics "z" value=0.815 
critical value = [-1.96001896  1.96001896]
z-statistic < z-critical value so we donot reject H0

alpha = 0.05
computed uncertainity p- value =0.41534189398889065
p-value> alpha so we donot reject H0
Decision = "Fails to reject H0"

One Sample Test for Proportion
Q: Three years back % of used cars with automatic transition is 23%.
Has it changed now?

Hypotheses Testing Steps 
Hypotheses | HO : pi = 0.23 HA : pi != 0.23 
Sample Statistics =  
Test Statistics "z" value=0.815 
critical value = ?

alpha = 0.05
computed uncertainity =?
Decision =?

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
#no. of gearbox type 'automatic' in the sample
count = sample['gearbox'].value_counts().get('automatic')
print("Number of automatic gearbox in the sample:", count)
#total number of records in the sample
nobs = len(sample['gearbox'])
#hypothesized proportion
p0 = 0.23
# sample proportion z- value
p_hat = count / nobs    
print("Sample Proportion:", p_hat)
#perform one sample z-test for proportions
z_stat, p_value = proportions_ztest(count, nobs, value=p0, alternative='two-sided', prop_var=False)
print("z-statistic:", z_stat)
print("p-value:", p_value)
#critical value for two-tailed test
critical_value = stats.norm.ppf([alpha/2, 1 - alpha/2]) 
print("Critical values:", critical_value)

Number of automatic gearbox in the sample: 214
Sample Proportion: 0.214
z-statistic: -1.233678008148831
p-value: 0.21732291189942932
Critical values: [-1.95996398  1.95996398]


TEST RESULT
Sample Statistics (Sample Proportion): 0.214 
Test Statistics z-statistic: -1.23
Critical values: [-1.95996398  1.95996398]
z-value lies between the critical value so we donot reject H0

alpha = 0.05
computed uncertainity p-value: 0.217
p-value> alpha so we donot reject H0
Decision = "Fails to reject H0"

2- SAMPLE TEST FOR MEAN  
Q.Is the mean price of cars that have run 30000-60000KM 
is the same for cars that has run 70000-90000KM ?

Hypotheses Testing Steps 
first do F-test for variance1 = variance 2
second do Weltch T-test depending on F-test results

T-Test
Hypotheses | HO : mean1 = mean2 HA : mean1 != mean2 
Sample Statistics: depends of variance in both samples the same.
Test Statistics : Weltch test for Unequal variance 
critical value = ?

alpha = 0.05
computed uncertainity =?
Decision =?

In [ ]:
km7_9 = cars[(cars['kilometer'] >= 70000) & (cars['kilometer'] <= 90000)]
km3_6 = cars[(cars['kilometer'] >= 30000) & (cars['kilometer'] <= 60000)]

sample1 = km7_9.sample(n=500, random_state=0)
sample2 = km3_6.sample(n=500, random_state=0)

# first Perform F-test to compare variances of two samples

print("sample1_var :", sample1.price.var())
print("sample2_var :", sample2.price.var())

#compute the f-statistic  if var1 > var2
F = sample1.price.var()/sample2.price.var()
print("F-statistic:", F)
df1 = len(sample1.price) - 1
df2 = len(sample2.price) - 1
print("p-value :",stats.f.cdf(F, df1, df2))
alpha = 0.05
#critical value for two-tailed test
critical_value = stats.f.ppf([alpha/2, 1 - alpha/2], df1, df2)
print("Critical values:", critical_value)



sample1_mean : 9450.59
sample2_mean : 14515.678
sample1_var : 86753098.35060078
sample2_var : 155442577.9462085
F-statistic: 0.5581038316324245
p-value : 5.0498268005393084e-11
Critical values: [0.83888578 1.1920574 ]


F- TEST RESULT

Hypothesis : HO : variance1 = variance2 HA : variance1 != variance2 
Sample Statistics: sample1_var : 86753098.35060078
                   sample2_var : 155442577.9462085
Test Statistics : F-statistic: 0.5581038316324245
Critical values: [0.83888578 1.1920574 ]
F-value lies outside the critical value so reject H0

alpha = 0.05
computed uncertainity p-value: 5.0498268005393084e-11
p-Value< alpha so reject H0

Decision = Reject H0

In [120]:
print("sample1_mean :", sample1.price.mean())
print("sample2_mean :", sample2.price.mean())

#perform t-test for unequal variance (Welch's t-test)
t_stat, p_value = stats.ttest_ind(sample1['price'], sample2['price'], equal_var=False)
print("t-statistic:", t_stat)   
print("p-value:", p_value)
#critical value for two-tailed test
df = ((sample1['price'].var()/len(sample1)) + (sample2['price'].var()/len(sample2)))**2 / \
     ( ((sample1['price'].var()/len(sample1))**2) / (len(sample1)-1) + ((sample2['price'].var()/len(sample2))**2) / (len(sample2)-1) )
print("Degrees of freedom:", df)
critical_value = stats.t.ppf([alpha/2, 1 - alpha/2], df)
print("Critical values:", critical_value)

sample1_mean : 9450.59
sample2_mean : 14515.678
t-statistic: -7.277610434526923
p-value: 7.258473522297715e-13
Degrees of freedom: 923.7016134521454
Critical values: [-1.96253552  1.96253552]


T - TEST RESULT

Hypotheses | HO : mean1 = mean2 HA : mean1 != mean2 
Sample Statistics: depends of variance in both samples the same.
Test Statistics : Weltch test for Unequal variance t-statistic: -7.277610434526923
Critical values: [-1.96253552  1.96253552]
t- statistics lies outside the critical value so reject H0

alpha = 0.05
computed uncertainity p-value: 7.258473522297715e-13
p-value> alpha so we donot reject H0
Decision = "Fails to reject H0"